In [2]:
# ONE-SHOT DRIVER (Model 2: plaquette bits) — rebuilds EVERYTHING, runs (alpha,beta,eps_bit) sweeps, writes CSV.
# No dependencies on prior cells. Paste into one Colab cell.
#
# WARNING: This is compute-heavy (build basis + build off-templates). It is the “all-at-once” version you asked for.

import itertools, math, csv
from collections import defaultdict
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float64
print("device:", device)

# -----------------------------
# Config
# -----------------------------
L = 2
alpha_list = [0.5, 1.0]
beta_list  = [0.3, 1.0, 3.0]
eps_list   = [0.0, 0.1, 0.2, 0.3, 0.5]
g = 1.0
m_lanczos = 650
k_out = 10
seed = 1

# -----------------------------
# SU(3) reps p+q<=2
# -----------------------------
REPS = [(0,0),(1,0),(0,1),(2,0),(0,2),(1,1)]
REP_ID = {rq:i for i,rq in enumerate(REPS)}
def dual(r): return (r[1], r[0])

def C2(p,q): return (p*p + q*q + p*q + 3*p + 3*q)/3.0
C2_table = {REP_ID[r]: C2(*r) for r in REPS}

# Truncated tensor product table
def trunc(components):
    return tuple(sorted({c for c in components if c in REP_ID}))

TP = {}
def add_tp(a,b,comps): TP[(a,b)] = trunc(comps)

add_tp((1,0),(1,0), [(2,0),(0,1)])
add_tp((1,0),(0,1), [(0,0),(1,1)])
add_tp((0,1),(0,1), [(0,2),(1,0)])
add_tp((2,0),(1,0), [(1,1)])
add_tp((2,0),(0,1), [(1,0)])
add_tp((0,2),(0,1), [(1,1)])
add_tp((0,2),(1,0), [(0,1)])
add_tp((1,1),(1,0), [(1,0),(2,0)])
add_tp((1,1),(0,1), [(0,1),(0,2)])
add_tp((1,1),(1,1), [(0,0),(1,1)])
add_tp((2,0),(0,2), [(0,0),(1,1)])
add_tp((2,0),(2,0), [])
add_tp((0,2),(0,2), [])
add_tp((2,0),(1,1), [(2,0)])
add_tp((0,2),(1,1), [(0,2)])

for (a,b), comps in list(TP.items()):
    if (b,a) not in TP:
        TP[(b,a)] = comps

def tp(a,b):
    if (a,b) in TP:
        return TP[(a,b)]
    da, db = dual(a), dual(b)
    if (da,db) in TP:
        return tuple(sorted(dual(r) for r in TP[(da,db)]))
    return tuple()

def truncated_tensor_closure(rep_list):
    if not rep_list:
        return {(0,0)}
    S = {rep_list[0]}
    for r in rep_list[1:]:
        S2 = set()
        for s in S:
            for c in tp(s,r):
                S2.add(c)
        S = S2
        if not S:
            return set()
    return S

# -----------------------------
# Lattice (2D torus)
# -----------------------------
def link_id(x,y,mu): return (x % L, y % L, mu)
V = [(x % L, y % L) for x in range(L) for y in range(L)]
E = [link_id(x,y,mu) for x in range(L) for y in range(L) for mu in (0,1)]
E_index = {e:i for i,e in enumerate(E)}
P = [(x % L, y % L) for x in range(L) for y in range(L)]
P_index = {p:i for i,p in enumerate(P)}
n_plaq = len(P)

def outgoing_edges(v):
    x,y = v
    return [link_id(x,y,0), link_id(x,y,1)]
def incoming_edges(v):
    x,y = v
    return [link_id(x-1,y,0), link_id(x,y-1,1)]

def vertex_gauss_ok(rep_on_links):
    for v in V:
        reps = []
        for e in outgoing_edges(v):
            reps.append(REPS[rep_on_links[E_index[e]]])
        for e in incoming_edges(v):
            reps.append(dual(REPS[rep_on_links[E_index[e]]]))
        if (0,0) not in truncated_tensor_closure(reps):
            return False
    return True

# Plaquette branching
FUND, AFUND = (1,0), (0,1)

def fuse_channels(r,f): return tp(r,f)

def apply_plaquette_branches(rep_on_links, p_xy):
    x,y = p_xy
    e1 = link_id(x,y,0)
    e2 = link_id(x+1,y,1)
    e3 = link_id(x,y+1,0)  # backward => AFUND
    e4 = link_id(x,y,1)    # backward => AFUND
    reps = [REPS[rep_on_links[E_index[e]]] for e in (e1,e2,e3,e4)]
    chans = [
        fuse_channels(reps[0], FUND),
        fuse_channels(reps[1], FUND),
        fuse_channels(reps[2], AFUND),
        fuse_channels(reps[3], AFUND),
    ]
    if any(len(c)==0 for c in chans):
        return []
    out = []
    for r1,r2,r3,r4 in itertools.product(*chans):
        rep2 = list(rep_on_links)
        rep2[E_index[e1]] = REP_ID[r1]
        rep2[E_index[e2]] = REP_ID[r2]
        rep2[E_index[e3]] = REP_ID[r3]
        rep2[E_index[e4]] = REP_ID[r4]
        out.append(tuple(rep2))
    return out

# -----------------------------
# Build bases
# -----------------------------
all_rep_ids = list(range(len(REPS)))
print("Building link basis...")
link_basis = []
for assign in itertools.product(all_rep_ids, repeat=len(E)):
    if vertex_gauss_ok(assign):
        link_basis.append(tuple(assign))
print("link basis size:", len(link_basis))

bit_basis = list(itertools.product([0,1], repeat=n_plaq))
print("Building extended basis...")
basis = [(st,bits) for st in link_basis for bits in bit_basis]
nb = len(basis)
print("extended basis size:", nb)
basis_index = {st:i for i,st in enumerate(basis)}

# Precompute diagonal pieces independent of beta/eps
bitcount = np.zeros(nb, dtype=np.float64)
Ediag = np.zeros(nb, dtype=np.float64)
for i,(links,bits) in enumerate(basis):
    bitcount[i] = float(sum(bits))
    Ediag[i] = 0.5*(g**2) * sum(C2_table[rid] for rid in links)

# Sparse helpers
def symmetrize_sparse(H):
    H = H.coalesce()
    HT = torch.sparse_coo_tensor(
        torch.stack([H.indices()[1], H.indices()[0]], dim=0),
        H.values(),
        H.shape,
        device=device,
        dtype=dtype
    ).coalesce()
    H2 = (H + HT).coalesce()
    H2 = torch.sparse_coo_tensor(H2.indices(), 0.5*H2.values(), H2.shape, device=device, dtype=dtype).coalesce()
    return H2

@torch.no_grad()
def lanczos_sparse_full_reorth(Hsparse, m=m_lanczos, k=k_out, seed=seed):
    torch.manual_seed(seed)
    n = Hsparse.shape[0]
    q = torch.randn(n, device=device, dtype=dtype)
    q = q / (q.norm() + 1e-18)
    Q = []
    alphaL = torch.zeros(m, device=device, dtype=dtype)
    betaL  = torch.zeros(m, device=device, dtype=dtype)
    q_prev = torch.zeros_like(q)
    for j in range(m):
        z = torch.sparse.mm(Hsparse, q.unsqueeze(1)).squeeze(1)
        a = torch.dot(q, z)
        z = z - a*q
        if j > 0:
            z = z - betaL[j-1]*q_prev
        for qi in Q:
            z = z - torch.dot(qi, z) * qi
        b = z.norm()
        alphaL[j] = a
        betaL[j] = b
        if b.item() < 1e-14:
            alphaL = alphaL[:j+1]
            betaL = betaL[:j]
            break
        Q.append(q)
        q_prev = q
        q = z / b
    a_cpu = alphaL.detach().cpu()
    b_cpu = betaL[:len(a_cpu)-1].detach().cpu()
    T = torch.diag(a_cpu)
    if len(b_cpu) > 0:
        T = T + torch.diag(b_cpu, 1) + torch.diag(b_cpu, -1)
    evals = torch.linalg.eigvalsh(T).numpy()
    evals.sort()
    return evals[:k]

def gap_from_list(evals):
    e0 = float(evals[0])
    for ev in evals[1:]:
        if float(ev) > e0 + 1e-10:
            return e0, float(ev - e0)
    return e0, float("nan")

# Pre-build diag index
diag_idx = torch.arange(nb, device=device, dtype=torch.int64)
diag_idx2 = torch.stack([diag_idx, diag_idx], dim=0)

# Off-template builder per alpha (unit beta, no diag)
def build_off_template(alpha):
    acc = defaultdict(float)
    for i,(links,bits) in enumerate(basis):
        for p_xy in P:
            pidx = P_index[p_xy]
            branches = apply_plaquette_branches(links, p_xy)
            if not branches:
                continue
            bits_toggled = list(bits)
            bits_toggled[pidx] = 1 - bits_toggled[pidx]
            bits_toggled = tuple(bits_toggled)

            js = []
            ws = []
            for links2 in branches:
                j = basis_index.get((links2, bits_toggled), None)
                if j is None:
                    continue
                s = 0.0
                for rid in links2:
                    s += C2_table[rid]
                w = math.exp(-alpha * s)
                js.append(j); ws.append(w)
            if not ws:
                continue
            Z = sum(ws)
            for j,w in zip(js,ws):
                acc[(i,j)] += -(w/Z)  # unit beta
    keys = list(acc.keys())
    rows = np.fromiter((k[0] for k in keys), dtype=np.int64, count=len(keys))
    cols = np.fromiter((k[1] for k in keys), dtype=np.int64, count=len(keys))
    vals = np.fromiter((acc[k] for k in keys), dtype=np.float64, count=len(keys))
    off_idx = torch.stack([torch.from_numpy(rows), torch.from_numpy(cols)], dim=0).to(device=device, dtype=torch.int64)
    off_val_unit = torch.from_numpy(vals).to(device=device, dtype=dtype)
    return off_idx, off_val_unit

results = []

for alpha in alpha_list:
    print(f"\n=== alpha={alpha} building off-template ===")
    off_idx, off_val_unit = build_off_template(alpha)
    print("off nnz =", off_val_unit.numel())

    for beta in beta_list:
        print(f"\n--- alpha={alpha} beta={beta} ---")
        off_val = beta * off_val_unit
        diag_base = Ediag + beta * n_plaq

        for eps_bit in eps_list:
            diag = diag_base + eps_bit * bitcount
            diag_val = torch.from_numpy(diag).to(device=device, dtype=dtype)

            idx_all = torch.cat([off_idx, diag_idx2], dim=1)
            val_all = torch.cat([off_val, diag_val], dim=0)

            Hs = torch.sparse_coo_tensor(idx_all, val_all, (nb, nb), device=device, dtype=dtype).coalesce()
            Hs = symmetrize_sparse(Hs)

            evals = lanczos_sparse_full_reorth(Hs)
            e0, gap = gap_from_list(evals)
            results.append((alpha, beta, eps_bit, e0, gap))
            print(f"eps={eps_bit:.1f} e0={e0:.6f} gap={gap:.6f}")

out_path = "/mnt/data/casrg_model2_results.csv"
with open(out_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["alpha","beta","eps_bit","e0","gap"])
    w.writerows(results)

print("\nWrote:", out_path)
print("Done.")


device: cuda
Building link basis...
link basis size: 3183
Building extended basis...
extended basis size: 50928

=== alpha=0.5 building off-template ===
off nnz = 192512

--- alpha=0.5 beta=0.3 ---
eps=0.0 e0=6.081618 gap=0.029348
eps=0.1 e0=6.205741 gap=0.008346
eps=0.2 e0=6.254211 gap=0.004469


KeyboardInterrupt: 